# ReAct 框架 - 第二部分：工具定义与使用

## 学习目标
1. 学会定义和注册工具
2. 掌握工具执行机制
3. 理解提示构建和解析

## 目录
1. [SimpleTool 工具类](#1-simpletool-工具类)
2. [工具注册与管理](#2-工具注册与管理)
3. [ReActPromptBuilder](#3-reactpromptbuilder)
4. [ReActParser](#4-reactparser)
5. [练习](#5-练习)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.react import (
    SimpleTool, ReActPromptBuilder, ReActParser,
    Thought, Action, Observation
)
print("模块加载成功！")

---
## 1. SimpleTool 工具类

### 1.1 创建基础工具

In [ ]:
# 定义计算器工具
def calculator_func(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"计算错误: {e}"

calculator = SimpleTool(
    name="calculator",
    description="执行数学计算，输入数学表达式",
    func=calculator_func
)

print(f"工具名称: {calculator.name}")
print(f"工具描述: {calculator.description}")

In [ ]:
# 执行工具
result = calculator.execute("15 * 24")
print(f"计算 15*24 = {result}")

result = calculator.execute("(100 + 50) / 3")
print(f"计算 (100+50)/3 = {result}")

### 1.2 创建搜索工具

In [ ]:
# 模拟搜索工具
def search_func(query: str) -> str:
    # 模拟搜索结果
    mock_results = {
        "python": "Python是一种高级编程语言，由Guido van Rossum创建",
        "北京天气": "北京今天晴，气温25°C",
        "机器学习": "机器学习是AI的一个分支，让计算机从数据中学习"
    }
    for key, value in mock_results.items():
        if key in query.lower():
            return value
    return f"未找到关于'{query}'的结果"

search = SimpleTool(
    name="search",
    description="搜索信息，输入搜索关键词",
    func=search_func
)

print(f"搜索'Python': {search.execute('Python')}")
print(f"搜索'北京天气': {search.execute('北京天气')}")

### 1.3 创建完成工具

In [ ]:
# 完成任务工具
def finish_func(answer: str) -> str:
    return f"任务完成，答案: {answer}"

finish = SimpleTool(
    name="finish",
    description="完成任务并返回最终答案",
    func=finish_func
)

print(finish.execute("2024年有8784小时"))

---
## 2. 工具注册与管理

### 2.1 工具集合

In [ ]:
# 创建工具集合
tools = [calculator, search, finish]

print("可用工具：")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

In [ ]:
# 工具查找函数
def get_tool(name: str, tools: list) -> SimpleTool:
    for tool in tools:
        if tool.name == name:
            return tool
    return None

# 测试
calc = get_tool("calculator", tools)
print(f"找到工具: {calc.name}")

### 2.2 工具描述生成

In [ ]:
def generate_tool_descriptions(tools: list) -> str:
    """生成工具描述文本"""
    lines = ["可用工具："]
    for tool in tools:
        lines.append(f"- {tool.name}: {tool.description}")
    return "\n".join(lines)

desc = generate_tool_descriptions(tools)
print(desc)

---
## 3. ReActPromptBuilder

### 3.1 基本使用

In [ ]:
# 创建提示构建器
builder = ReActPromptBuilder()

# 构建提示
prompt = builder.build(question="计算2024年有多少小时")

print("ReAct 提示：")
print("="*50)
print(prompt[:500])  # 显示前500字符

### 3.2 添加历史记录

In [ ]:
# 模拟历史记录
history = """Thought: 2024年是闰年，有366天
Action: calculator(366 * 24)
Observation: 8784"""

prompt_with_history = builder.build(
    question="计算2024年有多少小时",
    history=history
)

print("带历史的提示：")
print(prompt_with_history[:300])

---
## 4. ReActParser

### 4.1 解析思考

In [ ]:
parser = ReActParser()

# 解析思考
thought_text = "Thought: 我需要计算366天有多少小时"
thought = parser.parse_thought(thought_text)

print(f"解析思考: {thought}")

### 4.2 解析行动

In [ ]:
# 解析行动
action_text = "Action: calculator(366 * 24)"
result = parser.parse_action(action_text)

if result:
    tool_name, tool_input = result
    print(f"工具名称: {tool_name}")
    print(f"工具输入: {tool_input}")
else:
    print("无法解析行动")

In [ ]:
# 更多解析示例
actions = [
    "Action: search(北京天气)",
    "Action: finish(答案是8784小时)",
    "Action: calculator(100 / 4 + 25)"
]

print("解析多个行动：")
for action in actions:
    result = parser.parse_action(action)
    if result:
        name, inp = result
        print(f"  {name}({inp})")
    else:
        print(f"  无法解析: {action}")

### 4.3 完整解析流程

In [ ]:
# 模拟 LLM 输出
llm_output = """Thought: 我需要计算2024年有多少小时。2024年是闰年，有366天。
Action: calculator(366 * 24)"""

# 解析
lines = llm_output.strip().split('\n')
thought = parser.parse_thought(lines[0])
result = parser.parse_action(lines[1])

print("解析结果：")
print(f"  Thought: {thought}")
if result:
    tool_name, tool_input = result
    print(f"  Action: {tool_name}({tool_input})")

---
## 5. 练习

### 练习1：创建自定义工具

In [ ]:
# TODO: 创建一个"天气查询"工具
# def weather_func(city: str) -> str:
#     pass
# weather_tool = SimpleTool(...)

### 练习2：解析 LLM 输出

In [ ]:
# TODO: 解析以下 LLM 输出并执行工具
test_output = """Thought: 我需要搜索Python的信息
Action: search(Python编程语言)"""

# 你的代码...

---
## 下一步

继续学习 **02c_ReAct_Agent.ipynb** 了解完整的 ReAct Agent